In [65]:
from pyscf import gto, scf, cc
import numpy as np

a = 2.5 # 2aB
nH = 4
atoms = ""
for i in range(nH):
    atoms += f"H {i*a:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms, basis="6-31g", unit='B', spin=0, verbose=4)
mol.build()

mf = scf.RHF(mol)
mf.kernel()

mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)
mo = mf.stability()[0]
dm = mf.make_rdm1(mo,mf.mo_occ)
mf.kernel(dm0=dm)

nfrozen = 0
mycc = cc.CCSD(mf,frozen=nfrozen)
mycc.kernel()[0]
# et = mycc.ccsd_t()
# print(mycc.e_tot + et)

System: uname_result(system='Linux', node='yichi-thinkpad', release='4.4.0-26100-Microsoft', version='#7705-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 12
Python 3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:16:10) [GCC 13.3.0]
numpy 1.24.3  scipy 1.14.1  h5py 3.12.1
Date: Sat Feb 21 22:38:20 2026
PySCF version 2.8.0
PySCF path  /home/yichi/research/software/lno_pyscf
GIT HEAD (branch master) ef75f4190e4de208685670651dc6c467f72b6794

[ENV] PYSCF_EXT_PATH /home/yichi/research/software/pyscf
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 4
[INPUT] num. electrons = 4
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 H      0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000

-0.09001227608947138

In [66]:
def r2u_t1(rt1):
    return [rt1, rt1]

def r2u_t2(rt2):
    ut2aa = (rt2 - rt2.transpose(1,0,2,3))
    return [ut2aa, rt2, ut2aa]

In [67]:
ut1 = r2u_t1(mycc.t1)
ut2 = r2u_t2(mycc.t2)

In [ ]:
# t2_asy = mycc.t2
# # t2_asy = (t2_asy - t2_asy.transpose(1,0,2,3)) / 2
# # t2_asy = (t2_asy - t2_asy.transpose(0,1,3,2)) / 2
# # t2_asy = (t2_asy + t2_asy.transpose(1,0,3,2)) / 2
# # t2_asy = (t2_asy - t2_asy.transpose(0,1,3,2))
# t2_asy = (t2_asy - t2_asy.transpose(1,0,2,3))

In [68]:
myucc = cc.UCCSD(mf,frozen=nfrozen)
myucc.kernel(t1=ut1, t2=ut2)[0]


******** <class 'pyscf.cc.uccsd.UCCSD'> ********
CC2 = 0
CCSD nocc = (2, 2), nmo = (8, 8)
frozen orbitals 0
max_cycle = 50
direct = 0


conv_tol = 1e-07
conv_tol_normt = 1e-06
diis_space = 6
diis_start_cycle = 0
diis_start_energy_diff = 1e+09
max_memory 4000 MB (current use 176 MB)
Init E_corr(UCCSD) = -0.0900122760894713
cycle = 1  E_corr(UCCSD) = -0.0900122741511486  dE = 1.93832272e-09  norm(t1,t2) = 1.0447e-07
UCCSD converged
E(UCCSD) = -2.150311503234231  E_corr = -0.09001227415114862


-0.09001227415114862

In [69]:
mycct = cc.CCSDT(mf)
nocc, nmo = mycct.nocc, mycct.nmo
nvir = nmo - nocc
t1 = mycc.t1
t2 = mycc.t2
from math import prod, factorial
nx = lambda n, order: prod(n + i for i in range(order)) // factorial(order)
cc_order = mycct.cc_order
tamps = [t1, t2]

for order in range(2, cc_order - 1):
    t = np.zeros((nocc,) * (order + 1) + (nvir,) * (order + 1), dtype=t1.dtype)
    tamps.append(t)
if mycct.do_tri_max_t:
    t = np.zeros((nx(nocc, cc_order),) + (nvir,) * cc_order, dtype=t1.dtype)
else:
    t = np.zeros((nocc,) * cc_order + (nvir,) * cc_order, dtype=t1.dtype)
tamps.append(t)

#mycct = cc.CCSDT(mf)#, compact_tamps=True)
mycct.conv_tol = 1e-7
mycct.conv_tol_normt = 1e-6
mycct.level_shift = 0.2
mycct.verbose = 4
mycct.blksize = 1
mycct.set_einsum_backend('pyscf')
mycct.incore_complete = False
print(mycct.ao2mo)
mycct.kernel(tamps=tamps)

print("CCSDT Energy: ", mycct.e_tot)

AttributeError: module 'pyscf.cc' has no attribute 'CCSDT'

In [35]:
mycct.blksize

1

In [41]:
from pyscf import lib
from pyscf.cc import rccsdt

blksize = 3 #mycct.blksize
nvir = mycct.nmo - mycct.nocc
t3 = mycct.t3
t3_tmp = np.empty((blksize,) * 3 + (nvir,) * 3, dtype=t3.dtype)
# rccsdt._unpack_t3_(mycct, t3_tmp, )
for k0, k1 in lib.prange(0, nocc, blksize):
    bk = k1 - k0
    for j0, j1 in lib.prange(0, nocc, blksize):
        bj = j1 - j0
        for i0, i1 in lib.prange(0, nocc, blksize):
            bi = i1 - i0
            print(i0, i1, j0, j1, k0, k1)

nocc = mycct.nocc
t3_tmp = rccsdt._unpack_t3_(mycct, t3, t3_tmp, 0, nocc, 0, nocc, 0, nocc)

0 3 0 3 0 3


In [43]:
t3_tmp.shape

(3, 3, 3, 9, 9, 9)

In [46]:
# example for PT2
options = {'n_eql': 3,
           'n_prop_steps': 50,
            'n_ene_blocks': 1,
            'n_sr_blocks': 5,
            'n_blocks': 50,
            'n_walkers': 100,
            'seed': 2,
            'walker_type': 'rhf',
            'trial': 'ccsd_pt2',
            'dt':0.005,
            'free_projection':False,
            'fp_abs': False,
            'group': False,
            'ad_mode': None,
            'use_gpu': False,
            }

from ad_afqmc.prop_unrestricted import prep
import jax
jax.config.update("jax_enable_x64", True)
prep.prep_afqmc(mycc,chol_cut=1e-5)
# prop_unrestricted.run_afqmc(options,nproc=1)
option_file='options.bin'
import pickle
with open(option_file, 'wb') as f:
    pickle.dump(options, f)

#
# Preparing AFQMC calculation
# Calculating Cholesky integrals
# Finished calculating Cholesky integrals
#
# Size of the correlation space:
# Number of electrons: (3, 3)
# Number of basis functions: 12
# Number of Cholesky vectors: 36
#


In [47]:
import time
import numpy as np
from jax import random
from jax import numpy as jnp
from functools import partial 

ham_data, ham, prop, trial, wave_data, sampler, options = (prep._prep_afqmc())

init_time = time.time()

### initialize propagation
init_walkers = None
trial_rdm1 = trial.get_rdm1(wave_data)
if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial_rdm1
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)

prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers)
if jnp.abs(jnp.sum(prop_data["overlaps"])) < 1.0e-6:
    raise ValueError(
        "Initial overlaps are zero. Pass walkers with non-zero overlap."
    )
prop_data["key"] = random.PRNGKey(options["seed"])

prop_data["overlaps"] = trial.calc_overlap(prop_data["walkers"], wave_data)
prop_data["n_killed_walkers"] = 0

e_init= jnp.real(trial.calc_energy(prop_data["walkers"], ham_data, wave_data)[0])
# ept_sp = h0 + e0/t1 + e1/t1 - t2 * e0 / t1**2
# ept = jnp.array(jnp.sum(ept_sp) / prop.n_walkers)
prop_data["e_estimate"] = e_init
# eci = trial.calc_energy(
#     prop_data['walkers'], ham_data, wave_data)
prop_data["pop_control_ene_shift"] = prop_data["e_estimate"]

print(e_init)
print(e_init-mf.e_tot)

# norb: 12
# nelec: (3, 3)
# n_eql: 3
# n_prop_steps: 50
# n_ene_blocks: 1
# n_sr_blocks: 5
# n_blocks: 50
# n_walkers: 100
# seed: 2
# walker_type: rhf
# trial: ccsd_pt2
# dt: 0.005
# free_projection: False
# fp_abs: False
# group: False
# use_gpu: False
# n_exp_terms: 6
# symmetry: False
# save_walkers: False
# n_batch: 1
# max_error: 0.001
-3.0908772855484976
-4.5114160496240174e-07


In [ ]:
from jax import jit

@partial(jit, static_argnums=0)
def _calc_overlap_ci3(
    self,
    walker_up: jax.Array,
    walker_dn: jax.Array,
    ci3: jax.Array
    ) -> complex:
    noccA = self.nelec[0]
    noccB = self.nelec[1]
    ci3AAA, ci3AAB, ci3ABB, ci3BBB = ci3

    green_a = (walker_up.dot(jnp.linalg.inv(walker_up[: walker_up.shape[1], :]))).T
    green_b = (walker_dn.dot(jnp.linalg.inv(walker_dn[: walker_dn.shape[1], :]))).T
    green_a, green_b = green_a[:, noccA:], green_b[:, noccB:]
    # o0 = jnp.linalg.det(walker_up[:noccA, :]) * jnp.linalg.det(walker_dn[:noccB, :])
    # o1 = jnp.einsum("ia,ia", ci1A, green_a) + jnp.einsum("ia,ia", ci1B, green_b)
    # o2 = (
    #     0.5 * jnp.einsum("iajb, ia, jb", ci2AA, green_a, green_a)
    #     + 0.5 * jnp.einsum("iajb, ia, jb", ci2BB, green_b, green_b)
    #     + jnp.einsum("iajb, ia, jb", ci2AB, green_a, green_b)
    # )

    # ci3AAA = wave_data["ci3AAA"]
    # ci3AAB = wave_data["ci3AAB"]
    # ci3ABB = wave_data["ci3ABB"]
    # ci3BBB = wave_data["ci3BBB"]

    o3 = ( (1/6) * jnp.einsum("iajbkc, ia, jb, kc", ci3AAA, green_a, green_a, green_a)
         + (1/6) * jnp.einsum("iajbkc, ia, jb, kc", ci3BBB, green_b, green_b, green_b)
         + (1/2) * jnp.einsum("iajbkc, ia, jb, kc", ci3AAB, green_a, green_a, green_b)
         + (1/2) * jnp.einsum("iajbkc, ia, jb, kc", ci3ABB, green_a, green_b, green_b) ) 

    return o3